In [1]:
import pandas as pd
from collections import Counter

In [3]:
centrality = pd.read_csv('corpus_centrality.csv')

In [6]:
centrality = centrality[centrality['novel'] == 'Pride_and_Prejudice']

In [7]:
dial = centrality[centrality['graph_type'] == 'dialogue']
cooc = centrality[centrality['graph_type'] == 'cooccurrence']

In [21]:
dial.sort_values(by='pagerank', ascending=False).head(15)

,character,mentions,degree,strength,degree_centrality,betweenness,closeness,eigenvector,pagerank,novel,graph_type
16710,Elizabeth,493,25,386,0.694444,0.865079,2.333729,0.673459,0.258476,Pride_and_Prejudice,dialogue
16712,Mrs. Bennet,196,18,122,0.500000,0.125397,2.204032,0.241168,0.089259,Pride_and_Prejudice,dialogue
16711,Mr. Darcy,111,19,110,0.527778,0.056349,2.264770,0.375984,0.075873,Pride_and_Prejudice,dialogue
16715,Jane,108,13,104,0.361111,0.190476,2.276461,0.386792,0.074446,Pride_and_Prejudice,dialogue
16716,Miss Bingley,72,12,52,0.333333,0.211111,2.103608,0.107868,0.042530,Pride_and_Prejudice,dialogue
16717,Lydia,42,12,51,0.333333,0.000000,2.088276,0.144663,0.036732,Pride_and_Prejudice,dialogue
16713,Mr. Collins,46,15,40,0.416667,0.057937,2.013366,0.097045,0.035972,Pride_and_Prejudice,dialogue
16714,George Allen,48,14,48,0.388889,0.039683,2.168083,0.168738,0.034604,Pride_and_Prejudice,dialogue
16721,Wickham,54,7,52,0.194444,0.000000,2.224031,0.245672,0.034366,Pride_and_Prejudice,dialogue
16718,Mr. Bennet,43,11,44,0.305556,0.004762,2.001813,0.104532,0.032879,Pride_and_Prejudice,dialogue


In [20]:
cooc.sort_values(by='pagerank', ascending=False).head(10)

,character,mentions,degree,strength,degree_centrality,betweenness,closeness,eigenvector,pagerank,novel,graph_type
16523,Elizabeth,4335,120,1727,0.645161,0.718244,1.499970,0.604444,0.137910,Pride_and_Prejudice,cooccurrence
16527,Mr. Darcy,1808,69,878,0.370968,0.056641,1.493830,0.459599,0.066581,Pride_and_Prejudice,cooccurrence
16524,Jane,864,88,725,0.473118,0.098862,1.490752,0.331442,0.056971,Pride_and_Prejudice,cooccurrence
16526,Lady Catherine,541,75,507,0.403226,0.071764,1.482871,0.202985,0.042582,Pride_and_Prejudice,cooccurrence
16531,Mr. Bingley,690,59,507,0.317204,0.019641,1.484081,0.262321,0.039162,Pride_and_Prejudice,cooccurrence
16525,Lydia,415,76,440,0.408602,0.114136,1.481446,0.174768,0.037474,Pride_and_Prejudice,cooccurrence
16528,Mr. Collins,746,69,454,0.370968,0.057432,1.480828,0.175329,0.037093,Pride_and_Prejudice,cooccurrence
16530,Mrs. Bennet,707,60,448,0.322581,0.049278,1.475516,0.183729,0.035784,Pride_and_Prejudice,cooccurrence
16529,Wickham,607,61,397,0.327957,0.038834,1.481106,0.196506,0.032243,Pride_and_Prejudice,cooccurrence
16536,Miss Bingley,350,38,277,0.204301,0.010113,1.467543,0.142049,0.021196,Pride_and_Prejudice,cooccurrence


In [22]:
import networkx as nx
import pandas as pd
from pathlib import Path

def load_dc_graphs(graph_dir):
    graphs = {}
    for p in Path(graph_dir).glob("*_discussion.graphml"):
        novel = p.name.replace("_discussion.graphml", "")
        graphs[novel] = nx.read_graphml(p)
    return graphs

loaded_graphs = load_dc_graphs('../dc_graphs')

In [24]:
def extract_node_stats(G, novel):
    pr = nx.pagerank(G, weight="weight")
    rows = []

    for n in G.nodes():
        rows.append({
            "novel": novel,
            "character": n,
            "dc_in": G.nodes[n].get("dc_in", 0),
            "dc_out": G.nodes[n].get("dc_out", 0),
            "in_deg": G.in_degree(n, weight="weight"),
            "out_deg": G.out_degree(n, weight="weight"),
            "pagerank": pr.get(n, 0.0)
        })

    return pd.DataFrame(rows)

In [25]:
extracted_stats = []
for novel, G in loaded_graphs.items():
    stats_df = extract_node_stats(G, novel)
    extracted_stats.append(stats_df)
final_stats_df = pd.concat(extracted_stats, ignore_index=True)
final_stats_df.to_csv('dc_centrality.csv', index=False)

In [27]:
pnp = final_stats_df[final_stats_df['novel'] == 'Pride_and_Prejudice']

In [29]:
pnp.sort_values(by='in_deg', ascending=False).head(15)

,novel,character,dc_in,dc_out,in_deg,out_deg,pagerank
8070,Pride_and_Prejudice,Mr. Darcy,323,101,323,101,0.104042
8054,Pride_and_Prejudice,Elizabeth,311,524,311,524,0.122544
8003,Pride_and_Prejudice,Mr. Bingley,136,27,136,27,0.032980
8005,Pride_and_Prejudice,Wickham,129,55,129,55,0.040662
8072,Pride_and_Prejudice,Jane,108,133,108,133,0.029359
8048,Pride_and_Prejudice,Lady Catherine,85,64,85,64,0.026226
8019,Pride_and_Prejudice,Lydia,58,80,58,80,0.029398
8038,Pride_and_Prejudice,Mr. Collins,55,69,55,69,0.019950
8020,Pride_and_Prejudice,Mr. Bennet,39,56,39,56,0.009608
8027,Pride_and_Prejudice,Miss Bingley,33,83,33,83,0.011901


In [30]:
pnp.sort_values(by='out_deg', ascending=False).head(15)

,novel,character,dc_in,dc_out,in_deg,out_deg,pagerank
8054,Pride_and_Prejudice,Elizabeth,311,524,311,524,0.122544
8087,Pride_and_Prejudice,Mrs. Bennet,12,237,12,237,0.009945
8072,Pride_and_Prejudice,Jane,108,133,108,133,0.029359
8070,Pride_and_Prejudice,Mr. Darcy,323,101,323,101,0.104042
8027,Pride_and_Prejudice,Miss Bingley,33,83,33,83,0.011901
8019,Pride_and_Prejudice,Lydia,58,80,58,80,0.029398
8038,Pride_and_Prejudice,Mr. Collins,55,69,55,69,0.019950
8048,Pride_and_Prejudice,Lady Catherine,85,64,85,64,0.026226
8033,Pride_and_Prejudice,Mrs. Gardiner,3,63,3,63,0.006956
8020,Pride_and_Prejudice,Mr. Bennet,39,56,39,56,0.009608


In [31]:
MALE = {"he", "him", "his"}
FEMALE = {"she", "her", "hers"}

GENDERED = MALE | FEMALE

def coref_to_gender(entities_path, min_gendered = 2, min_share = 0.7):
    ents = pd.read_csv(entities_path, sep="\t", quoting=3)

    # Keep pronoun person mentions
    e = ents[(ents["cat"] == "PER") & (ents["prop"] == "PRON")].copy()
    if e.empty:
        return {}

    e["p"] = e["text"].astype(str).str.lower().str.strip()
    e = e[e["p"].isin(GENDERED)]
    if e.empty:
        return {}

    out = {}
    for coref, g in e.groupby("COREF"):
        m = g["p"].isin(MALE).sum()
        f = g["p"].isin(FEMALE).sum()
        tot = m + f

        if tot < min_gendered:
            out[str(coref)] = "UNK"
            continue

        share = max(m, f) / tot
        if share < min_share:
            out[str(coref)] = "UNK"
        else:
            out[str(coref)] = "M" if m > f else "F"

    return out

def top_proper_noun(df):
    prop = df.loc[df["prop"] == "PROP", "text"].dropna()
    vc = prop.value_counts()
    if len(vc):
        return vc.idxmax()
    vc = df["text"].dropna().value_counts()
    return vc.idxmax() if len(vc) else None

def coref_to_name(entities_path, min_mentions= 1):
    ents = pd.read_csv(entities_path, sep="\t", quoting=3)
    ents_per = ents[ents["cat"] == "PER"].copy()
    ents_per["COREF"] = ents_per["COREF"].astype(str)

    cluster_summary = (
        ents_per.groupby("COREF")
        .apply(lambda df: pd.Series({
            "n_mentions": len(df),
            "n_prop": (df["prop"] == "PROP").sum(),
            "top_text": top_proper_noun(df)
        }), include_groups=False)
        .reset_index()
    )

    clean = cluster_summary[
        (cluster_summary["n_prop"] > 0) &
        (cluster_summary["n_mentions"] >= min_mentions)
    ].dropna(subset=["top_text"])

    return dict(zip(clean["COREF"].astype(str), clean["top_text"]))
